In [1]:

import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# 1. Load dataset
df = pd.read_csv("../smart_campus_facility_dataset.csv")

# 2. Extract hour from complaint time
df["Complaint_Hour"] = pd.to_datetime(
    df["Complaint_Time"], format="%H:%M", errors="coerce"
).dt.hour

# 3. Select input features
features = [
    "User_Type",
    "Block_No",
    "Floor",
    "Room_No",
    "Facility_Type",
    "Issue_Type",
    "Day_of_Week",
    "Footfall",
    "Previous_Complaints",
    "Photo_Available",
    "Complaint_Hour"
]

X = df[features]
y = df["Severity"]

# 4. Separate categorical and numerical columns
categorical_features = [
    "User_Type",
    "Facility_Type",
    "Issue_Type",
    "Day_of_Week",
    "Photo_Available"
]

numerical_features = [
    "Block_No",
    "Floor",
    "Room_No",
    "Footfall",
    "Previous_Complaints",
    "Complaint_Hour"
]

# 5. Convert categorical values into numbers
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        ("numerical", "passthrough", numerical_features)
    ]
)

# 6. Split into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 7. Create the Random Forest model
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=150,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

# 8. Train the model
model.fit(X_train, y_train)

# 9. Evaluate the model
y_pred = model.predict(X_test)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred, labels=[
    "Low", "Medium", "High", "Critical"
]))

# 10. Save the trained model
joblib.dump(model, "../smart_campus_severity_model.pkl")

print("\nModel saved successfully!")


Training rows: 8000
Testing rows: 2000
Accuracy: 0.5325

Classification Report:
              precision    recall  f1-score   support

    Critical       0.53      0.29      0.37        80
        High       0.65      0.71      0.68       414
         Low       0.50      0.48      0.49       655
      Medium       0.50      0.51      0.50       851

    accuracy                           0.53      2000
   macro avg       0.54      0.50      0.51      2000
weighted avg       0.53      0.53      0.53      2000


Confusion Matrix:
[[315 340   0   0]
 [314 433 104   0]
 [  0 100 294  20]
 [  0   0  57  23]]

Model saved successfully!
